## Roll Rate

In [2]:
import pandas as pd
!pip install seaborn
import seaborn as sns

In [3]:
# Long format: одна строка = один кредит × один месяц
df = pd.DataFrame({
    'loan_id':    [100001, 100001, 100001, 100001,
                   100002, 100002, 100002,
                   100003, 100003, 100003],
    'mob':        [1, 2, 3, 4,
                   1, 2, 3,
                   1, 2, 3],
    'dpd_bucket': ['Current', 'Current', '1-30', '31-60',
                   'Current', '1-30', 'Current',
                   '1-30', '31-60', '61-90']
})
df


,loan_id,mob,dpd_bucket
0,100001,1,Current
1,100001,2,Current
2,100001,3,1-30
3,100001,4,31-60
4,100002,1,Current
5,100002,2,1-30
6,100002,3,Current
7,100003,1,1-30
8,100003,2,31-60
9,100003,3,61-90


In [ ]:
df = df.sort_values(['loan_id', 'mob']).reset_index(drop=True)


# Текущее состояние (месяц t)
from_state = df[['loan_id', 'mob', 'dpd_bucket']].rename(
    columns={'dpd_bucket': 'from_bucket'}
)

# Следующее состояние (месяц t+1)
to_state = df[['loan_id', 'mob', 'dpd_bucket']].copy()
to_state['mob'] = to_state['mob'] - 1  # сдвигаем на 1 назад
to_state = to_state.rename(columns={'dpd_bucket': 'to_bucket'})

# Объединяем по loan_id и mob
pairs = from_state.merge(to_state, on=['loan_id', 'mob'], how='inner')



print(pairs)


# Упорядочим бакеты
order = ['Current', '1-30', '31-60', '61-90', '90+']

# Строим матрицу переходов
roll_matrix = pd.crosstab(
    pairs['from_bucket'],
    pairs['to_bucket'],
    normalize='index'  # проценты по строкам
) * 100

# Приводим к нужному порядку
roll_matrix = roll_matrix.reindex(index=order, columns=order, fill_value=0)

print(roll_matrix.round(2))

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(roll_matrix, annot=True, fmt='.1f', cmap='RdYlGn_r')
plt.xlabel('TO (Куда перешёл)')
plt.ylabel('FROM (Откуда пришёл)')
plt.title('Roll Rate Matrix (%)')
plt.show()